# Imports

In [20]:
import pandas as pd
import numpy as np
import pgeocode

# Data

In [3]:
df = pd.read_parquet('./data/pp-complete.parquet')
df.head()

,price,date,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type
0,166500,1995-11-22,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A
1,59000,1995-09-27,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A
2,118000,1995-12-15,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A
3,48500,1995-01-27,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A
4,27500,1995-04-20,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A


In [4]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
first_date = df['date'].min()
df['passed'] = (df['date'] - first_date).dt.days

df.drop(columns=['date'], inplace=True)

df.head()

,price,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,year,month,passed
0,166500,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,1995,11,325
1,59000,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,1995,9,269
2,118000,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,1995,12,348
3,48500,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,1995,1,26
4,27500,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,1995,4,109


# Functions

In [25]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Все аргументы в градусах, возвращает расстояние в километрах.
    """
    # переводим в радианы
    φ1, φ2 = np.radians(lat1), np.radians(lat2)
    Δφ = np.radians(lat2 - lat1)
    Δλ = np.radians(lon2 - lon1)

    a = np.sin(Δφ/2)**2 + np.cos(φ1)*np.cos(φ2)*np.sin(Δλ/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    R = 6371.0  # средний радиус Земли в км

    return R * c

# Feature Engineering

In [ ]:
district_counts = df['district'].value_counts().reset_index()
district_counts.columns = ['district', 'count']

# Получаем список названий district, где число упоминаний < 100
low_count_districts = district_counts[district_counts['count'] < 100]['district'].tolist()

# Удаляем эти районы из основного датафрейма
df = df[~df['district'].isin(low_count_districts)]

df['median_price_all_type'] = df.groupby(['type', 'old_new', 'duration', 'ppd_type', 'district', 'year', 'month'])['price'].transform('median')
df.head(10)

,price,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,year,month,passed,median_price_all_type
0,166500,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,1995,11,325,140000.0
1,59000,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,1995,9,269,55000.0
2,118000,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,1995,12,348,106000.0
3,48500,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,1995,1,26,48000.0
4,27500,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,1995,4,109,43000.0
5,114000,SE24 0DH,S,N,F,13,BRANTWOOD ROAD,LONDON,LONDON,LAMBETH,GREATER LONDON,A,1995,6,180,95750.0
6,46000,TS10 2DJ,S,N,F,2,MALCOLM GROVE,REDCAR,REDCAR,LANGBAURGH-ON-TEES,CLEVELAND,A,1995,9,262,44500.0
7,52500,SN2 2SY,S,N,F,11,MARIGOLD CLOSE,SWINDON,SWINDON,THAMESDOWN,THAMESDOWN,A,1995,5,131,55250.0
8,19000,PO32 6EP,T,N,F,17,CLARENCE ROAD,EAST COWES,EAST COWES,ISLE OF WIGHT,ISLE OF WIGHT,A,1995,10,302,37750.0
9,67500,SE18 2DH,T,N,F,75,HIGHMEAD,LONDON,LONDON,GREENWICH,GREATER LONDON,A,1995,11,320,55000.0


In [18]:
nomi = pgeocode.Nominatim("gb")

# запрос сразу по вектору — вернёт DataFrame с полями latitude/longitude и прочими
geo = nomi.query_postal_code(df["postcode"].tolist())

df["latitude"]  = geo["latitude"].values
df["longitude"] = geo["longitude"].values

df.drop(columns=['postcode'], inplace=True)
df.dropna(subset=["latitude", "longitude"], inplace=True)

df.head()

,price,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,year,month,passed,median_price_all_type,latitude,longitude
0,166500,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,1995,11,325,140000.0,51.895390,0.156180
1,59000,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,1995,9,269,55000.0,53.432800,-2.909600
2,118000,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,1995,12,348,106000.0,51.502114,-0.551257
3,48500,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,1995,1,26,48000.0,52.478075,-1.447600
4,27500,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,1995,4,109,43000.0,52.832861,-2.965928


In [ ]:
#TODO Расчет расстояния до ближайшего города
#TODO Расчет расстояния до ближайшего крупного города
#TODO Расчет населения/числа продаж города
#TODO Добавить дискреты is_big_city/is_crysis
#TODO Добавить сопутствующие экономические метрики: ср.зарплата/инфляция/безработица

In [22]:
from geopy.geocoders import Nominatim
import time

# Инициализируем геокодер
geolocator = Nominatim(user_agent="my_geocoder")

# Прогоним все уникальные города
city_coords = {}
nn = 0
for city in df['town_city'].unique():
    try:
        nn += 1
        loc = geolocator.geocode(f"{city}, UK")
        if loc:
            city_coords[city] = (loc.latitude, loc.longitude)
            print(f"Добавлен город {nn}/{len(df['town_city'].unique())}: {city}")
        else:
            city_coords[city] = (None, None)
            print(f"Ошибка при добавлении города {nn}/{len(df['town_city'].unique())}: {city}")
    except Exception:
        city_coords[city] = (None, None)
    #time.sleep(1)  # Нужная задержка между запросами (1 запрос в секунду)

Добавлен город 1/1172: BISHOP'S STORTFORD
Добавлен город 2/1172: LIVERPOOL
Добавлен город 3/1172: SLOUGH
Добавлен город 4/1172: BEDWORTH
Добавлен город 5/1172: OSWESTRY
Добавлен город 6/1172: LONDON
Добавлен город 7/1172: REDCAR
Добавлен город 8/1172: SWINDON
Добавлен город 9/1172: EAST COWES
Добавлен город 10/1172: STAFFORD
Добавлен город 11/1172: DAGENHAM
Добавлен город 12/1172: CREWE
Добавлен город 13/1172: HALIFAX
Добавлен город 14/1172: CREWKERNE
Добавлен город 15/1172: WORTHING
Добавлен город 16/1172: WORCESTER
Добавлен город 17/1172: LEEDS
Добавлен город 18/1172: WETHERBY
Добавлен город 19/1172: OAKHAM
Добавлен город 20/1172: DURHAM
Добавлен город 21/1172: WIGAN
Добавлен город 22/1172: BEDFORD
Добавлен город 23/1172: ROCHESTER
Добавлен город 24/1172: EAST GRINSTEAD
Добавлен город 25/1172: LLANRWST
Добавлен город 26/1172: LEICESTER
Добавлен город 27/1172: LINCOLN
Добавлен город 28/1172: BRISTOL
Добавлен город 29/1172: YORK
Добавлен город 30/1172: BRACKNELL
Добавлен город 31/1172:

In [23]:
# Распаковываем city_coords в колонки
df['city_lat'] = df['town_city'].map(lambda c: city_coords[c][0])
df['city_lon'] = df['town_city'].map(lambda c: city_coords[c][1])

In [26]:
# Считаем расстояние
df['dist_km'] = haversine(
    df['city_lat'], df['city_lon'],
    df['latitude'], df['longitude']
)

df[['town_city','latitude','longitude','city_lat','city_lon','dist_km']].head()

,town_city,latitude,longitude,city_lat,city_lon,dist_km
0,BISHOP'S STORTFORD,51.895390,0.156180,51.867628,0.163196,3.124343
1,LIVERPOOL,53.432800,-2.909600,53.407199,-2.991680,6.139023
2,SLOUGH,51.502114,-0.551257,51.503427,-0.574872,1.641050
3,BEDWORTH,52.478075,-1.447600,52.479283,-1.466320,1.274898
4,OSWESTRY,52.832861,-2.965928,52.860310,-3.054820,6.704670


In [ ]:
cols = ["town_city", "city_lat", "city_lon", ]
df.to_csv("output_with_coords.csv", columns=cols, index=False)